In [104]:
from msi_visual.extraction import get_extraction_mz_list
from msi_visual.normalization import spatial_total_ion_count, total_ion_count, median_ion
from msi_visual.supervised.annotations import get_img, get_visualization, get_dataset
from scipy.stats import mannwhitneyu
import numpy as np
import tqdm
from argparse import Namespace
from pathlib import Path
import joblib
array = np.array

path = r"E:\MSImaging-data\_msi_visual\Extractions\atlas_verification"
extraction_args = eval(
    open(
        Path(path) /
        "args.txt").read())
extraction_mzs = extraction_args.mzs

paths = [(path + "\\0.npy", 1), (path + "\\2.npy", 0), (path + "\\1.npy", 2), (path + "\\3.npy", 3)]


X, y, label_encoder = get_dataset(r"../scripts/graph_data/NRL4485-s2_reannotation_23-12-24_PAHJ.json", paths, subsample=1, normalization=total_ion_count)
y = np.array(y)

data = {"extraction_args": extraction_args, "X": X, "y": y, "label_encoder": label_encoder}
joblib.dump(data, "raw_data.joblib")


using normalization <function total_ion_count at 0x0000022A30716160>
using normalization <function total_ion_count at 0x0000022A30716160>
using normalization <function total_ion_count at 0x0000022A30716160>
using normalization <function total_ion_count at 0x0000022A30716160>


['raw_data.joblib']

In [55]:
from sklearn.metrics import roc_auc_score
import joblib

auc_mzs = joblib.load("auc_mzs.joblib")
top_shap_mzs = joblib.load("shap_mzs_tic.joblib")

categories = list(auc_mzs["common"].keys())
unique_mzs = auc_mzs["unique"]
common_mzs = auc_mzs["common"]

auc_scores, shap_scores, intersection_scores = [], [], []
ratios = []
mzs_per_category = {}
for category in categories:
    if top_shap_mzs[category] is None:
        top_shap_mzs[category] = []
        continue

    # Get unique and common m/z values with scores
    unique_pairs_orig = sorted(unique_mzs[category], key=lambda x: x[1], reverse=True)
    common_pairs_orig = sorted(common_mzs[category], key=lambda x: x[1], reverse=True)
    auc_pairs = sorted(unique_mzs[category] + common_mzs[category], key=lambda x: x[1], reverse=True)
    intersection_mzs = [x[0] for x in auc_pairs if x[0] in top_shap_mzs[category][-50 : ]]
    mzs_per_category[category] = intersection_mzs

In [ ]:
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import ClippedAdam

# -------------------
# Prepare your data
# -------------------
# Replace with your actual data
X_torch = torch.tensor(X, dtype=torch.float32)  # [N, D]
y_torch = torch.tensor(y, dtype=torch.long)     # [N]
N, D = X_torch.shape
C = int(y_torch.max().item()) + 1
G = 5  # Number of latent groups

pyro.clear_param_store()
pyro.set_rng_seed(0)

# -------------------
# Model
# -------------------
def model(X, y=None):
    with pyro.plate("groups", G):
        shared_w = pyro.sample("shared_w", dist.Normal(0, 1).expand([D]).to_event(1))  # [G, D]

    group_probs = pyro.sample("group_probs", dist.Dirichlet(torch.ones(G)))  # [G]

    with pyro.plate("categories", C):
        group_id = pyro.sample("group_id", dist.Categorical(group_probs))  # latent group assignment
        specific_w = pyro.sample("specific_w", dist.Normal(0, 0.5).expand([D]).to_event(1))  # region-specific
        w = shared_w[group_id] + specific_w  # final per-region weights

    with pyro.plate("data", N):
        logits = torch.matmul(X, w.T)  # [N, C]
        pyro.sample("obs", dist.Categorical(logits=logits), obs=y)

# -------------------
# Guide
# -------------------
def guide(X, y=None):
    shared_loc = pyro.param("shared_loc", torch.zeros(G, D))
    shared_scale = pyro.param("shared_scale", torch.ones(G, D), constraint=dist.constraints.positive)
    with pyro.plate("groups", G):
        pyro.sample("shared_w", dist.Normal(shared_loc, shared_scale).to_event(1))

    group_logits = pyro.param("group_logits", torch.zeros(G))
    pyro.sample("group_probs", dist.Dirichlet(torch.exp(group_logits)))

    category_group_logits = pyro.param("category_group_logits", torch.randn(C, G))
    specific_loc = pyro.param("specific_loc", torch.zeros(C, D))
    specific_scale = pyro.param("specific_scale", torch.ones(C, D), constraint=dist.constraints.positive)

    with pyro.plate("categories", C):
        pyro.sample("group_id", dist.Categorical(logits=category_group_logits))
        pyro.sample("specific_w", dist.Normal(specific_loc, specific_scale).to_event(1))

# -------------------
# Train the model
# -------------------
optimizer = ClippedAdam({"lr": 1e-3})
svi = SVI(model, guide, optimizer, loss=Trace_ELBO())

for step in range(20000 * 10):
    loss = svi.step(X_torch, y_torch)
    if step % 1000 == 0:
        print(f"Step {step} - Loss: {loss:.2f}")

# -------------------
# Extract results
# -------------------
shared_w_mean = pyro.param("shared_loc").detach().cpu().numpy()         # [G, D]
specific_w_mean = pyro.param("specific_loc").detach().cpu().numpy()     # [C, D]
group_assignments = torch.softmax(pyro.param("category_group_logits"), dim=-1).detach().cpu().numpy()  # [C, G]


Step 0 - Loss: 12252170.90


KeyboardInterrupt: 

In [85]:
shared_w_mean = pyro.param("shared_loc").detach().cpu().numpy()         # [G, D]
specific_w_mean = pyro.param("specific_loc").detach().cpu().numpy()     # [C, D]
group_assignments = torch.softmax(pyro.param("category_group_logits"), dim=-1).detach().cpu().numpy()  # [C, G]


category = 102  # example

# soft group assignment: shape [G]
group_probs_c = group_assignments[category]  # soft probabilities for this category

# compute expected shared_w: weighted sum of all G shared vectors
shared_w_c = (group_probs_c[:, None] * shared_w_mean).sum(axis=0)  # shape: [D]

w_c = shared_w_c + specific_w_mean[category]  # shape: [D]

top_20_mz_idx = (w_c).argsort()[::-1][:50]  # largest magnitude values

mzs = [extraction_mzs[i] for i in top_20_mz_idx]
print(mzs)

[885.54829752, 885.55444354, 886.55296545, 857.517149, 857.51706004, 887.55554101, 858.52058908, 883.53274826, 858.51944986, 658.85612769, 834.5282567, 909.54899223, 882.5276895, 884.53634704, 881.51653566, 865.50307848, 888.5595747, 888.5585549, 859.52412677, 1179.73081311, 910.55275137, 882.530108, 540.0534138, 1180.73943434, 866.50730465, 786.52859303, 426.02157169, 786.52904411, 943.50723857, 857.56876675, 858.5690201, 884.58690604, 884.58859563, 1032.54608696, 885.59773493, 840.5308999, 832.51198242, 837.54153533, 911.55845223, 965.51728982, 346.056369, 819.51638903, 911.56261141, 966.51723411, 795.51954846, 833.51553325, 837.55398369, 690.50717159, 338.98963316, 1181.74247752]


mzs_per_category["PLQ_ARTEFACT_ARTEFACT"]

In [65]:
mzs_per_category["PLQ_ARTEFACT_ARTEFACT"]

[1207.75938587,
 480.30869475,
 1180.73943434,
 1207.78445573,
 540.0534138,
 865.50307848,
 866.50730465,
 1179.73081311,
 857.51706004,
 858.51944986,
 858.52058908,
 859.52412677]

In [39]:
category_names = label_encoder.inverse_transform(y)
print([(i, y[i]) for i in range(len(category_names)) if "PLQ" in category_names[i]])

[(10782, 102), (10783, 102), (10784, 102), (10785, 102), (10786, 102), (10787, 102), (10788, 102), (10789, 102), (10790, 102), (10791, 102), (10792, 102), (10793, 102), (10794, 102), (10795, 102), (10796, 102), (10797, 102), (10798, 102), (10799, 102), (10800, 102), (10801, 102), (10802, 102), (10803, 102), (10804, 102), (10805, 102), (10806, 102), (10807, 102), (10808, 102), (10809, 102), (10810, 102), (10811, 102), (10812, 102), (10813, 102), (10814, 102), (10815, 102), (10816, 102), (10817, 102), (10818, 102), (10819, 102), (10820, 102), (10821, 102), (10822, 102), (10823, 102), (10824, 102), (10825, 102), (10826, 102), (10827, 102), (10828, 102), (10829, 102), (10830, 102), (10831, 102), (10832, 102), (10833, 102), (10834, 102), (10835, 102), (10836, 102), (10837, 102), (10838, 102), (10839, 102), (10840, 102), (10841, 102), (10842, 102), (10843, 102), (10844, 102), (10845, 102), (10846, 102), (10847, 102), (10848, 102), (10849, 102), (10850, 102), (10851, 102), (10852, 102), (1085

In [99]:
def get_top_mz_for_category(c, top_k=20):
    """
    Get top m/z indices for category c based on learned weights.

    Parameters:
    - c (int): category index
    - top_k (int): number of top m/z values to return

    Returns:
    - top_indices (ndarray): indices of top m/z values
    - w_c (ndarray): full weight vector for category c
    """
    # 1. Get soft group probabilities for category c
    group_probs_c = group_assignments[c]  # shape: [G]

    # 2. Compute expected shared weight for category c
    shared_w_c = (group_probs_c[:, None] * shared_w_mean).sum(axis=0)  # shape: [D]

    # 3. Add category-specific component
    w_c = shared_w_c + specific_w_mean[c]  # total weight for category c

    # 4. Get indices of top_k m/z values (by absolute magnitude)
    top_indices = (w_c).argsort()[::-1][:top_k]

    return top_indices, w_c

def get_top_mz_by_mean_minus_std(c, top_k=20):
    """
    Rank m/z by mean - std for category `c`.
    Returns top indices and associated values.
    """
    # Get soft group assignment
    group_probs = group_assignments[c]  # [G]
    shared_mean = (group_probs[:, None] * shared_w_mean).sum(axis=0)
    specific_mean = specific_w_mean[c]
    w_c_mean = shared_mean + specific_mean

    # Compute std via law of total variance (as in previous function)
    shared_std = pyro.param("shared_scale").detach().cpu().numpy()
    shared_var = shared_std**2
    E_var = (group_probs[:, None] * shared_var).sum(axis=0)
    Var_mean = (group_probs[:, None] * (shared_w_mean - shared_mean)**2).sum(axis=0)
    shared_total_var = E_var + Var_mean
    specific_std = pyro.param("specific_scale")[c].detach().cpu().numpy()
    specific_var = specific_std**2
    w_c_var = shared_total_var + specific_var
    w_c_std = np.sqrt(w_c_var)

    # Compute score: mean - std
    score = (w_c_mean) - w_c_std
    top_indices = np.argsort(score)[::-1][:top_k]

    return top_indices, score[top_indices]


print([(i, y[i]) for i in range(len(category_names)) if "PLQ" in category_names[i]])
mapped_mz = {}
for label_index, name in enumerate(label_encoder.classes_):
    indices, _ = get_top_mz_by_mean_minus_std(label_index, 150)
    mzs = [extraction_mzs[i] for i in indices]
    mapped_mz[name] = mzs

joblib.dump(mapped_mz, "bayes_mzs.joblib")

[(10782, 87), (10783, 87), (10784, 87), (10785, 87), (10786, 87), (10787, 87), (10788, 87), (10789, 87), (10790, 87), (10791, 87), (10792, 87), (10793, 87), (10794, 87), (10795, 87), (10796, 87), (10797, 87), (10798, 87), (10799, 87), (10800, 87), (10801, 87), (10802, 87), (10803, 87), (10804, 87), (10805, 87), (10806, 87), (10807, 87), (10808, 87), (10809, 87), (10810, 87), (10811, 87), (10812, 87), (10813, 87), (10814, 87), (10815, 87), (10816, 87), (10817, 87), (10818, 87), (10819, 87), (10820, 87), (10821, 87), (10822, 87), (10823, 87), (10824, 87), (10825, 87), (10826, 87), (10827, 87), (10828, 87), (10829, 87), (10830, 87), (10831, 87), (10832, 87), (10833, 87), (10834, 87), (10835, 87), (10836, 87), (10837, 87), (10838, 87), (10839, 87), (10840, 87), (10841, 87), (10842, 87), (10843, 87), (10844, 87), (10845, 87), (10846, 87), (10847, 87), (10848, 87), (10849, 87), (10850, 87), (10851, 87), (10852, 87), (10853, 87), (10854, 87), (10855, 87), (10856, 87), (10857, 87), (10858, 87)

['bayes_mzs.joblib']

In [91]:
for g in range(10):
    print(g, [(index, label_encoder.classes_[index]) for index, a in enumerate(np.argmax(group_assignments, axis=1)) if a == g])

0 [(5, 'BST_HY_RCH'), (20, 'BST_MB_SNC'), (26, 'BST_MB_SNR SUB6'), (27, 'BST_MB_VTA'), (29, 'BST_TH_ICL'), (35, 'BST_TH_MG'), (38, 'BST_TH_PF'), (52, 'CNU_BST_PR'), (53, 'CNU_CP_ARTEFACT'), (55, 'CNU_GPE_GPE SUB1'), (59, 'CNU_GPE_GPE SUB5'), (61, 'CNU_SAMY_AAA'), (67, 'CTX_EPD_ARTEFACT'), (69, 'CTX_HPF_CA1-SLM'), (77, 'CTX_HPF_CA3-SLM'), (79, 'CTX_HPF_CA3-SO'), (85, 'CTX_HPF_DG-PO'), (87, 'CTX_HPF_GC-PO'), (97, 'CTX_L4_ARTEFACT'), (100, 'CTX_PIR_PIR L2'), (105, 'WM_ALV_ARTEFACT'), (108, 'WM_CING_ARTEFACT'), (109, 'WM_CPD_ARTEFACT'), (112, 'WM_EC_ARTEFACT'), (114, 'WM_FR_ARTEFACT'), (122, 'WM_ST_ARTEFACT')]
1 [(0, 'BG_ARTEFACT_ARTEFACT'), (15, 'BST_MB_OP'), (23, 'BST_MB_SNR SUB3'), (24, 'BST_MB_SNR SUB4'), (30, 'BST_TH_LGD'), (34, 'BST_TH_MD L'), (36, 'BST_TH_MGD'), (40, 'BST_TH_POL'), (42, 'BST_TH_SGN'), (43, 'BST_TH_SPFM'), (46, 'BST_TH_VPL'), (54, 'CNU_CP_CP STRIAE'), (64, 'CNU_SI_ARTEFACT'), (66, 'CTX_CTXSP_EPD'), (73, 'CTX_HPF_CA1-SP'), (75, 'CTX_HPF_CA2+3-SLM'), (80, 'CTX_HPF_CA3-

In [50]:
def get_shared_mz_between_categories(c1, c2, top_k=20):
    """
    Return top shared m/z values between two categories c1 and c2
    based on similarity of their expected shared weights.
    """

    # 1. Get expected shared weights for each category
    w1_shared = (group_assignments[c1][:, None] * shared_w_mean).sum(axis=0)  # shape [D]
    w2_shared = (group_assignments[c2][:, None] * shared_w_mean).sum(axis=0)  # shape [D]

    # 2. Measure agreement: similarity or overlap of high-importance m/z
    avg_shared = (w1_shared + w2_shared) / 2
    diff = np.abs(w1_shared - w2_shared)

    # 3. Score = strong average weight, low disagreement
    score = (avg_shared) / (diff + 1e-6)

    # 4. Top m/z indices with strongest shared signal
    top_shared_idx = np.argsort(score)[::-1][:top_k]

    return top_shared_idx, w1_shared[top_shared_idx], w2_shared[top_shared_idx], avg_shared[top_shared_idx]

c1, c2 = 102, 86
top_mz, w1_vals, w2_vals, avg_vals = get_shared_mz_between_categories(c1, c2)

for mz_idx, v1, v2, v_avg in zip(top_mz, w1_vals, w2_vals, avg_vals):
    print(f"m/z {extraction_mzs[mz_idx]} | cat {c1}: {v1:.3f} | cat {c2}: {v2:.3f} | avg: {v_avg:.3f}")


m/z 905.57388203 | cat 102: 0.041 | cat 86: 0.041 | avg: 0.041
m/z 812.59247 | cat 102: 0.021 | cat 86: 0.021 | avg: 0.021
m/z 812.5212 | cat 102: 0.008 | cat 86: 0.008 | avg: 0.008
m/z 854.57083927 | cat 102: 0.031 | cat 86: 0.031 | avg: 0.031
m/z 892.56711451 | cat 102: 0.018 | cat 86: 0.018 | avg: 0.018
m/z 700.52772046 | cat 102: 0.007 | cat 86: 0.007 | avg: 0.007
m/z 809.47582779 | cat 102: 0.056 | cat 86: 0.057 | avg: 0.056
m/z 480.27230486 | cat 102: 0.099 | cat 86: 0.099 | avg: 0.099
m/z 847.66002209 | cat 102: 0.028 | cat 86: 0.029 | avg: 0.028
m/z 943.53429065 | cat 102: 0.016 | cat 86: 0.016 | avg: 0.016
m/z 812.5701 | cat 102: 0.020 | cat 86: 0.020 | avg: 0.020
m/z 1113.90041301 | cat 102: 0.041 | cat 86: 0.041 | avg: 0.041
m/z 360.94226871 | cat 102: 0.016 | cat 86: 0.016 | avg: 0.016
m/z 890.57811284 | cat 102: 0.035 | cat 86: 0.035 | avg: 0.035
m/z 1115.91204394 | cat 102: 0.044 | cat 86: 0.044 | avg: 0.044
m/z 812.5614 | cat 102: 0.014 | cat 86: 0.015 | avg: 0.015
m/z 8

In [53]:
def get_top_shared_mz_for_group(g, top_k=20):
    shared_w_g = shared_w_mean[g]  # shape [D]
    top_mz_idx = (shared_w_g).argsort()[::-1][:top_k]
    return top_mz_idx, shared_w_g[top_mz_idx]

indices, w = get_top_shared_mz_for_group(1)
mzs = [extraction_mzs[i] for i in indices]
print(mzs)

[528.29385508, 1003.50490255, 812.5437, 889.65163077, 848.47565574, 812.55707, 724.49974481, 717.52716978, 812.51245, 812.56146, 480.27230486, 826.52545482, 1115.91204394, 574.2368581, 812.51227, 849.64186762, 796.57549934, 890.57811284, 812.5348, 1180.77183556]


In [100]:
mapped_mz["PLQ_ARTEFACT_ARTEFACT"]

[885.54829752,
 885.55444354,
 886.55296545,
 857.517149,
 857.51706004,
 887.55554101,
 834.5282567,
 858.52058908,
 858.51944986,
 883.53274826,
 658.85612769,
 909.54899223,
 882.5276895,
 884.53634704,
 881.51653566,
 865.50307848,
 888.5585549,
 888.5595747,
 859.52412677,
 1179.73081311,
 910.55275137,
 882.530108,
 540.0534138,
 1180.73943434,
 866.50730465,
 786.52859303,
 426.02157169,
 943.50723857,
 786.52904411,
 858.5690201,
 884.58690604,
 857.56876675,
 884.58859563,
 1032.54608696,
 837.54153533,
 911.55845223,
 885.59773493,
 911.56261141,
 840.5308999,
 832.51198242,
 795.51954846,
 819.51638903,
 837.55398369,
 346.056369,
 833.51553325,
 929.56625467,
 1181.74247752,
 966.51723411,
 338.98963316,
 690.50717159,
 1033.54549732,
 1207.75938587,
 1003.47066935,
 856.54366138,
 889.56157921,
 856.54196326,
 965.51728982,
 817.50718178,
 885.57728355,
 771.53294285,
 702.54109733,
 905.57034483,
 773.53236668,
 905.57388203,
 376.94518276,
 859.57488311,
 505.98822211,
 